# ☕ Dirty Cafe Sales: A Complete Data Cleaning Journey

**Goal:** Our dataset contains 10,000 records of cafe sales, but heavily suffers from missing data, "UNKNOWN" string values, and missing dates. Instead of blindly dropping rows and losing over 70% of the dataset, we will use logical deduction and straightforward algebra to rescue as much data as possible.

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset, automatically parsing known bad string values to NaN
df = pd.read_csv("./dataset/dirty_cafe_sales.csv", na_values=["UNKNOWN", "ERROR"])

print(f"Initial Shape: {df.shape}")
print(f"Rows with at least one NaN: {df.isna().any(axis=1).sum()}")
display(df.head())

## 1. Correcting Data Types
Pandas initially reads columns with missing values as generic text (objects). Before doing any math, we must convert numeric columns to floats and the date column to a proper datetime type.

In [ ]:
# Convert math columns to floats
df["Quantity"] = df["Quantity"].astype("float")
df["Price Per Unit"] = df["Price Per Unit"].astype("float")
df["Total Spent"] = df["Total Spent"].astype("float")

# Convert Transaction Date string to a datetime object
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], format="%Y-%m-%d", errors="coerce")

print(df.dtypes)

## 2. Early Sanity Checks
Before deducing missing values, we must prove our existing numbers are valid. Let's check for negative quantities, zero-prices, and decimal quantities.

In [ ]:
invalid_qty = df["Quantity"].le(0).sum()
decimal_qty = (df["Quantity"].notna() & (df["Quantity"] % 1 != 0)).sum()
invalid_prices = df["Price Per Unit"].le(0).sum()
invalid_totals = df["Total Spent"].le(0).sum()

print("--- Sanity Check Results ---")
print(f"Zero/Negative Quantities: {invalid_qty}")
print(f"Decimal Quantities: {decimal_qty}")
print(f"Zero/Negative Prices: {invalid_prices}")
print(f"Zero/Negative Totals: {invalid_totals}")

# Result: 0 errors found! Existing numeric data is clean.

## 3. Data Rescue Phase 1: Menu Mapping
Based on provided cafe context, our menu has strict prices. 
We can use a dictionary to fill missing `Price Per Unit` values if we know the `Item`. 

*Note: Cake and Juice both cost $3, and Sandwich and Smoothie both cost $4. If an Item is missing but the price is $3 or $4, we cannot safely guess which item it is, so we will ignore those ambiguous prices for now.*

In [ ]:
item_to_price = {
    "Cookie": 1.0,
    "Tea": 1.5,
    "Coffee": 2.0,
    "Cake": 3.0,
    "Juice": 3.0,
    "Sandwich": 4.0,
    "Smoothie": 4.0,
    "Salad": 5.0,
}

# Reverse mapping for unique prices only
price_to_item = {
    1.0: "Cookie",
    1.5: "Tea",
    2.0: "Coffee",
    # 3.0: Ambiguous (Cake or Juice)
    # 4.0: Ambiguous (Sandwich or Smoothie)
    5.0: "Salad",
}

# 1. Fill missing Prices using valid Items
valid_item_mask = df["Item"].notna() & df["Price Per Unit"].isna()
df.loc[valid_item_mask, "Price Per Unit"] = df.loc[valid_item_mask, "Item"].map(item_to_price)

# 2. Fill missing Items using unique Prices
valid_unique_price_mask = df["Item"].isna() & df["Price Per Unit"].isin(price_to_item.keys())
df.loc[valid_unique_price_mask, "Item"] = df.loc[valid_unique_price_mask, "Price Per Unit"].map(price_to_item)

print(f"Remaining Missing Items: {df['Item'].isna().sum()}")
print(f"Remaining Missing Prices: {df['Price Per Unit'].isna().sum()}")

## 4. Data Rescue Phase 2: Mathematical Deduction
Every transaction must follow the exact rule: `Total Spent = Quantity * Price Per Unit`. 
Using algebra, if exactly *one* of these three pillars is missing for a row, we can calculate it perfectly using the other two.

In [ ]:
# Deduction masks
missing_total = df["Total Spent"].isna() & df["Quantity"].notna() & df["Price Per Unit"].notna()
missing_qty = df["Quantity"].isna() & df["Total Spent"].notna() & df["Price Per Unit"].notna()
missing_price = df["Price Per Unit"].isna() & df["Total Spent"].notna() & df["Quantity"].notna()

# Execute math calculations
df.loc[missing_total, "Total Spent"] = df.loc[missing_total, "Quantity"] * df.loc[missing_total, "Price Per Unit"]
df.loc[missing_qty, "Quantity"] = df.loc[missing_qty, "Total Spent"] / df.loc[missing_qty, "Price Per Unit"]
df.loc[missing_price, "Price Per Unit"] = df.loc[missing_price, "Total Spent"] / df.loc[missing_price, "Quantity"]

print("--- After Math Deduction ---")
print(f"Missing Quantities: {df['Quantity'].isna().sum()}")
print(f"Missing Prices: {df['Price Per Unit'].isna().sum()}")
print(f"Missing Totals: {df['Total Spent'].isna().sum()}")

## 5. Data Rescue Phase 3: Secondary Mapping
Thanks to our mathematical deductions, we recovered dozens of new `Price Per Unit` values. We can now run our unique price-to-item mapping a second time to catch any newly discovered items!

In [ ]:
# Run the reverse mapping one more time
valid_unique_price_mask2 = df["Item"].isna() & df["Price Per Unit"].isin(price_to_item.keys())
df.loc[valid_unique_price_mask2, "Item"] = df.loc[valid_unique_price_mask2, "Price Per Unit"].map(price_to_item)

print(f"Missing Items reduced to: {df['Item'].isna().sum()}")

## 6. Verifying the Deductions
Let's guarantee that our logic didn't introduce any mismatches (like a Cookie costing $5, or broken math equations in a row).

In [ ]:
# Check 1: Price-Item Mismatch
mismatches = df.dropna(subset=["Item", "Price Per Unit"])
bad_prices = mismatches[mismatches["Price Per Unit"] != mismatches["Item"].map(item_to_price)]
print(f"Item/Price Mismatches found: {len(bad_prices)}")

# Check 2: Algebraic Mismatch
math_mismatches = df.dropna(subset=["Quantity", "Price Per Unit", "Total Spent"])
broken_math = math_mismatches[math_mismatches["Total Spent"] != (math_mismatches["Quantity"] * math_mismatches["Price Per Unit"])]
print(f"Algebraic Mismatches found: {len(broken_math)}")

## 7. Handling Remaining Unresolvable Data
After careful extraction, we have hit the limits of 100% true deductive reasoning. For the remaining data, we must decide to Impute (guess using statistics) or Drop.

1. **Location & Payment Method**: Missing at very high rates (~30%). Filling with guessing algorithms injects false bias since the data is missing uniformly. We will fill them honestly with **"Unspecified"**.
2. **Item**: The remaining missing items fall directly into the ambiguous $3 and $4 price buckets. Because the Item is the most critical feature, we will drop these.
3. **Transaction Dates**: We cannot perform time-series analysis with missing dates, so these row are dropped.
4. **Quantity/Total**: Rows missing *both* metrics are mathematically dead ends. We drop them.

In [ ]:
# 1. Fill heavily missing categories safely
df.fillna({"Location": "Unspecified", "Payment Method": "Unspecified"}, inplace=True)

# 2. Drop ambiguous Items
df.dropna(subset=["Item"], inplace=True)

# 3. Drop unresolvable Dates
df.dropna(subset=["Transaction Date"], inplace=True)

# 4. Drop mathematically unresovable equations
df.dropna(subset=["Quantity", "Total Spent"], inplace=True)

print("Final Missing Value Check:")
display(df.isna().sum())

## 8. Final Polish & Export
We finish up by casting Quantity to integer, dropping duplicate rows, and preparing for export!

In [ ]:
# Final formatting
df["Quantity"] = df["Quantity"].astype(int)
df = df.drop_duplicates()
df.reset_index(drop=True, inplace=True)

print(f"Final Cleaned Dataset Shape: {df.shape}")
print(f"Total rows salvaged from absolute deletion: {df.shape[0] - (10000 - 6911)}")

# Save to Kaggle output root
df.to_csv("./dataset/cleaned_cafe_sales.csv", index=False)
display(df.head())